In [1]:
!pip install python-docx pandas tqdm

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 4.0/4.0 MB 34.6 MB/s  0:00:00

   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   ---------------------------------------- 0/2 [lxml]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   -------------------- ------------------- 1/2 [python-docx]
   ---------------------------------------- 2/2 [python-docx]



In [2]:
from docx import Document
from pathlib import Path
import pandas as pd
import re
from tqdm import tqdm


In [3]:
CHAPTER_PATTERN = re.compile(r'^제\s*\d+\s*장')
ARTICLE_PATTERN = re.compile(r'^제\s*\d+\s*조')  # DOCX에서는 이 정도면 충분
CLAUSE_PATTERN = re.compile(r'^[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]\s*')

In [4]:
def split_clause(line: str):
    line = line.strip()
    m = CLAUSE_PATTERN.match(line)
    if not m:
        return None, line
    clause_no = m.group(0).strip()
    rest = CLAUSE_PATTERN.sub("", line).strip()
    return clause_no, rest

In [5]:
def split_article_title(line: str):
    """
    '제1조(목적)' → ('제1조', '목적')
    '제10조 수용 및 사용' → ('제10조', '수용 및 사용')
    """
    m = re.match(r'^제(\d+)조(?:\((.*?)\))?\s*(.*)$', line)
    if not m:
        return None, None, None
    num, title_paren, title_after = m.groups()
    article_number = f"제{num}조"

    if title_paren:
        return article_number, title_paren.strip(), None
    if title_after:
        return article_number, title_after.strip(), None

    return article_number, None, None

In [6]:
def parse_law_docx(docx_path: str, law_name: str):
    doc = Document(docx_path)
    records = []

    current_chapter = None
    current_article_number = None
    current_article_title = None
    current_clause_number = None

    clause_buffer = []
    article_head_buffer = []

    def flush_clause():
        nonlocal clause_buffer, current_clause_number
        if clause_buffer:
            records.append({
                "law_name": law_name,
                "chapter": current_chapter,
                "article_number": current_article_number,
                "article_title": current_article_title,
                "clause_number": current_clause_number,
                "text": " ".join(clause_buffer).strip()
            })
        clause_buffer = []

    def flush_article_head():
        nonlocal article_head_buffer
        if article_head_buffer:
            records.append({
                "law_name": law_name,
                "chapter": current_chapter,
                "article_number": current_article_number,
                "article_title": current_article_title,
                "clause_number": None,
                "text": " ".join(article_head_buffer).strip()
            })
        article_head_buffer = []

    for para in doc.paragraphs:
        line = para.text.strip()
        if not line:
            continue

        # 장
        if CHAPTER_PATTERN.match(line):
            flush_clause()
            flush_article_head()
            current_chapter = line
            continue

        # 조
        if ARTICLE_PATTERN.match(line):
            flush_clause()
            flush_article_head()

            art_no, art_title, body_inline = split_article_title(line)
            current_article_number = art_no
            current_article_title = art_title
            current_clause_number = None

            if body_inline:
                # 조 제목과 본문이 같은 줄인 경우
                article_head_buffer.append(body_inline)
            continue

        # 항(①…)
        clause_no, rest = split_clause(line)
        if clause_no:
            flush_clause()
            current_clause_number = clause_no
            clause_buffer = [rest] if rest else []
            continue

        # 본문
        if current_clause_number:
            clause_buffer.append(line)
        else:
            article_head_buffer.append(line)

    # 마지막 flush
    flush_clause()
    flush_article_head()

    df = pd.DataFrame(records)
    return df

In [7]:
def process_all_docx(input_dir: str, output_dir: str, merge=True):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    docx_files = list(input_dir.glob("*.docx"))

    all_dfs = []

    for path in tqdm(docx_files):
        law_name = path.stem
        df = parse_law_docx(str(path), law_name)
        df.to_csv(output_dir / f"{law_name}.csv",
                  index=False, encoding="utf-8-sig")
        all_dfs.append(df)

    if merge:
        merged = pd.concat(all_dfs, ignore_index=True)
        merged.to_csv(output_dir / "ALL_LAWS.csv",
                      index=False, encoding="utf-8-sig")

In [8]:
INPUT_DIR = "./law_doc_downloads"
OUTPUT_DIR = "./laws_csv"

process_all_docx(INPUT_DIR, OUTPUT_DIR)


  0%|                                                                                          | 0/148 [00:00<?, ?it/s]


PackageNotFoundError: Package not found at 'law_doc_downloads\간선급행버스체계의 건설 및 운영에 관한 특별법.docx'

In [9]:
# !pip install pywin32

import win32com.client as win32
from pathlib import Path

def convert_doc_to_docx(input_dir: str):
    input_dir = Path(input_dir)
    word = win32.gencache.EnsureDispatch('Word.Application')
    word.Visible = False

    files = list(input_dir.glob("*.doc"))
    print(f"총 {len(files)}개의 .doc 파일 발견")

    for file in files:
        docx_path = file.with_suffix(".docx")
        if docx_path.exists():
            print(f"[SKIP] 이미 존재: {docx_path.name}")
            continue

        print(f"[변환 중] {file.name} → {docx_path.name}")
        doc = word.Documents.Open(str(file))
        doc.SaveAs(str(docx_path), FileFormat=16)  # 16 = wdFormatXMLDocument (.docx)
        doc.Close()

    word.Quit()
    print("=== 변환 완료 ===")


NameError: name 'cwd' is not defined

In [11]:
convert_doc_to_docx("law_doc_downloads")


총 15개의 .doc 파일 발견
[변환 중] 건축법 시행령(대통령령)(제35811호)(20251001).doc → 건축법 시행령(대통령령)(제35811호)(20251001).docx


com_error: (-2147352567, '예외가 발생했습니다.', (0, 'Microsoft Word', '파일을 찾을 수 없습니다. 파일이 이동 또는 삭제되었거나 이름이 변경되었는지 확인하세요.\r ("C:\\...\\건축법 시행령(대통령령)(제35811호)(20251...")', 'wdmain11.chm', 24654, -2146823114), None)

In [13]:
import re
from pathlib import Path

def sanitize_doc_filename(path: Path) -> Path:
    """
    너무 긴 .doc 파일명을 Word가 읽을 수 있도록 짧게 변환.
    괄호 제거 + 공백 → _ + 50자 제한.
    """
    name = path.stem
    # 괄호 제거
    name = re.sub(r'\(.*?\)', '', name)
    # 공백 제거
    name = name.replace(' ', '_')
    # 길이 제한
    name = name[:50]

    new_path = path.with_name(name + path.suffix)
    path.rename(new_path)
    return new_path


In [14]:
from pathlib import Path

DOWNLOAD_DIR = Path("law_doc_downloads")

for f in DOWNLOAD_DIR.glob("*.doc"):
    new_path = sanitize_doc_filename(f)
    print("변경:", f.name, "→", new_path.name)


변경: 건축법 시행령(대통령령)(제35811호)(20251001).doc → 건축법_시행령.doc
변경: 건축법(법률)(제21065호)(20251001).doc → 건축법.doc
변경: 국토의 계획 및 이용에 관한 법률 시행령(대통령령)(제35628호)(20251002).doc → 국토의_계획_및_이용에_관한_법률_시행령.doc
변경: 도로교통법(법률)(제20677호)(20250722).doc → 도로교통법.doc
변경: 먹는물관리법(법률)(제21065호)(20251001).doc → 먹는물관리법.doc
변경: 산업집적활성화 및 공장설립에 관한 법률(법률)(제21065호)(20251001).doc → 산업집적활성화_및_공장설립에_관한_법률.doc
변경: 상법(법률)(제20991호)(20250722).doc → 상법.doc
변경: 식품위생법 시행령(대통령령)(제35811호)(20251001).doc → 식품위생법_시행령.doc
변경: 식품위생법(법률)(제21065호)(20251001).doc → 식품위생법.doc
변경: 의료법 시행규칙(보건복지부령)(제01116호)(20250621).doc → 의료법_시행규칙.doc
변경: 자동차관리법 시행규칙(국토교통부령)(제01519호)(20250814).doc → 자동차관리법_시행규칙.doc
변경: 자동차관리법(법률)(제21065호)(20251001).doc → 자동차관리법.doc
변경: 자본시장과 금융투자업에 관한 법률 시행령(대통령령)(제35872호)(20251125).doc → 자본시장과_금융투자업에_관한_법률_시행령.doc
변경: 자전거 이용 활성화에 관한 법률(법률)(제19162호)(20230704).doc → 자전거_이용_활성화에_관한_법률.doc
변경: 장애인복지법(법률)(제20929호)(20251023).doc → 장애인복지법.doc
